# IndustrialData Mini · Sistemas de Big Data · RA1
## 00 · El encargo (minutos 0–18)

**Solución del profesor.** Datos ficticios: dos máquinas, cuatro sensores, diez minutos. Pregunta: ¿qué máquina consume menos por pieza y con qué fiabilidad podemos afirmarlo?

Antes de la sesión: `bash preparar.sh` y `bash iniciar.sh` en Terminal de macOS. En clase no descargaremos imágenes. Ejecuta celdas con **Shift+Enter**. Sigue 00 → 05. Cada cuaderno tiene sus propios imports; no depende de variables de otro kernel.

La carga ordinaria funciona también sin notebook: `docker compose run --rm pipeline`. Los cuadernos explican el proceso; no son una dependencia del panel. No edites `data/raw`. Para experimentar, utiliza copias o argumentos distintos en memoria.

In [ ]:
import json
import pandas as pd
from industrialdata.common import root
from industrialdata.catalog import load_catalog
from industrialdata.ingest import ingest, business_tables
folder = root()
catalog = load_catalog(folder)  # Consulta PostgreSQL: NO el catálogo JSON de pruebas.
from industrialdata.catalog import connection
with connection() as conn:
    for sql in ["SELECT * FROM machines ORDER BY machine_id",
                "SELECT sensor_id, machine_id, variable, unit, operational_max FROM sensors ORDER BY sensor_id"]:
        cur = conn.execute(sql)
        display(pd.DataFrame(cur.fetchall(), columns=[c.name for c in cur.description]))

## Cuatro fuentes, distintos tipos de información
Sensores: JSON Lines. Producción y mantenimiento: CSV. Catálogo: SQL en PostgreSQL. El JSON `catalog.json` es una instantánea auxiliar y permite tests offline; el recorrido de clase consulta PostgreSQL.

In [ ]:
rows = [json.loads(line) for line in (folder / "raw/telemetry_001.jsonl").read_text().splitlines()]
print("Recepciones:", len(rows))
display(pd.DataFrame(rows).head())
display(pd.read_csv(folder / "raw/production.csv"))
display(pd.read_csv(folder / "raw/maintenance.csv"))

## Claves y conjuntos
Una fila de telemetría no representa una máquina entera: es la observación de un sensor en un instante. La clave semántica es `(sensor_id, timestamp_utc)`. Un sensor pertenece a una máquina; una máquina tiene varios sensores.

In [ ]:
catalogadas = {m["machine_id"] for m in catalog["machines"]}
recibidas = {row["machine_id"] for row in rows}
print("Conocidas:", recibidas & catalogadas)
print("Huérfanas:", recibidas - catalogadas)
assert recibidas - catalogadas == {"M999"}

**Pregunta al grupo:** ¿debemos borrar una temperatura de 85 °C? No si es físicamente plausible: puede ser una incidencia, no un error de datos.

**Pizarra:** fuentes → validación → observaciones fiables → indicadores → panel. En este caso no se añade Hadoop, Kafka ni nube; se mantiene la arquitectura del trabajo guiado.